In [1]:
import pandas as pd
import numpy as np


In [2]:
ratings = pd.read_csv('../data/ml-32m/ratings.csv')

In [3]:
print(f"Dataset Size: {ratings.memory_usage(index=True).sum() / 1024**2} MB")

Dataset Size: 976.5688514709473 MB


In [4]:
ratings.dtypes

userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object

In [5]:
ratings.userId.max()

np.int64(200948)

In [6]:
# Set userId to be uint32 to save memory
ratings['userId'] = ratings['userId'].astype(np.uint32)

Ratings

In [7]:
ratings.rating.min()

np.float64(0.5)

In [8]:
ratings.rating.max()

np.float64(5.0)

In [9]:
# Set rating to be half since it can only take values from 0.5 to 5.0 in 0.5 increments
ratings['rating'] = ratings['rating'].astype(np.float16)

MovieId

In [10]:
print(ratings.movieId.max())

292757


In [11]:
# Set movieId to be uint32 to save memory
ratings['movieId'] = ratings['movieId'].astype(np.uint32)

Timestamp

In [12]:
print(ratings.timestamp.max())

1697164147


In [13]:
# Set timestamp to be uint32 to save memory
ratings['timestamp'] = ratings['timestamp'].astype(np.uint32)

### Size after setting correct dtypes

In [14]:
print(f"Dataset Size: {ratings.memory_usage(index=True).sum() / 1024**2} MB")

Dataset Size: 427.2489433288574 MB


# Make Implicit User-Item Matrix

Create a `scipy` sparse matrix to save spaces.
- Using Pandas `pivot` requires ~32GB of RAM
- `scipy` saves only the nonzero entries
- Matrix can be saved as `.npz` file

<font color="red"> Unused for now </font> because dataset is explicit feedback and that is the first test.

In [15]:
from scipy.sparse import csr_matrix

# Map userId and movieId to contiguous 0-based indices
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}

row = ratings['userId'].map(user_to_idx).values
col = ratings['movieId'].map(movie_to_idx).values
data = np.ones(len(ratings), dtype=np.int8)  # implicit feedback: 1 = rated

user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(movie_ids)))

print(f"Shape: {user_item_matrix.shape}")
print(f"Non-zero entries: {user_item_matrix.nnz:,}")
print(f"Sparsity: {1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6f}")
print(f"Memory usage: {(user_item_matrix.data.nbytes + user_item_matrix.indices.nbytes + user_item_matrix.indptr.nbytes) / 1024**2:.2f} MB")

Shape: (200948, 84432)
Non-zero entries: 32,000,204
Sparsity: 0.998114
Memory usage: 153.36 MB


Save matrix

In [17]:
from scipy.sparse import save_npz
save_npz('../data/ml-32m/implicit_binary_user_item_matrix.npz', user_item_matrix)

# Make Explicit User-Item Matrix

In [17]:
from scipy.sparse import csr_matrix
from scipy.sparse import save_npz

user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}

row = ratings['userId'].map(user_to_idx).values
col = ratings['movieId'].map(movie_to_idx).values

data = ratings['rating'].values.astype(np.float32)  # use actual ratings
user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(movie_ids)))
print(f"Shape: {user_item_matrix.shape}")
print(f"Non-zero entries: {user_item_matrix.nnz:,}")
print(f"Sparsity: {1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.6f}")
print(f"Memory usage: {(user_item_matrix.data.nbytes + user_item_matrix.indices.nbytes + user_item_matrix.indptr.nbytes) / 1024**2:.2f} MB")

save_npz('../data/ml-32m/explicit_user_item_matrix.npz', user_item_matrix)

Shape: (200948, 84432)
Non-zero entries: 32,000,204
Sparsity: 0.998114
Memory usage: 244.91 MB


But that's just the full matrix. We need train and test sets.

## Training Set
- For each user, leave their last interaction out of the train set to be used in test set

In [18]:
def pop_last_user_review(ratings_df):
    # Sort by timestamp to ensure the last review is at the end
    ratings_df = ratings_df.sort_values(by='timestamp')
    
    # Get the last review for each user
    last_reviews = ratings_df.groupby('userId').tail(1)
    
    # Remove the last reviews from the original DataFrame
    ratings_without_last = ratings_df[~ratings_df.index.isin(last_reviews.index)]
    
    return ratings_without_last, last_reviews

In [19]:
for user_id, group in ratings.groupby('userId'):
    print(f"User {user_id} has {len(group)} reviews")
ratings_without_last, last_reviews = pop_last_user_review(ratings)
print(f"Ratings without last review: {ratings_without_last.shape[0]}")
print(f"Last reviews: {last_reviews.shape[0]}")
ratings_without_last.to_csv('../data/ml-32m/ratings_without_last.csv', index=False)

User 1 has 141 reviews
User 2 has 52 reviews
User 3 has 147 reviews
User 4 has 27 reviews
User 5 has 33 reviews
User 6 has 26 reviews
User 7 has 44 reviews
User 8 has 31 reviews
User 9 has 58 reviews
User 10 has 660 reviews
User 11 has 20 reviews
User 12 has 23 reviews
User 13 has 65 reviews
User 14 has 29 reviews
User 15 has 82 reviews
User 16 has 295 reviews
User 17 has 86 reviews
User 18 has 138 reviews
User 19 has 47 reviews
User 20 has 140 reviews
User 21 has 35 reviews
User 22 has 63 reviews
User 23 has 63 reviews
User 24 has 45 reviews
User 25 has 87 reviews
User 26 has 33 reviews
User 27 has 62 reviews
User 28 has 2842 reviews
User 29 has 151 reviews
User 30 has 21 reviews
User 31 has 67 reviews
User 32 has 44 reviews
User 33 has 265 reviews
User 34 has 123 reviews
User 35 has 503 reviews
User 36 has 125 reviews
User 37 has 235 reviews
User 38 has 29 reviews
User 39 has 57 reviews
User 40 has 101 reviews
User 41 has 37 reviews
User 42 has 48 reviews
User 43 has 150 reviews
User

## Test Set
- The left-one-out ratings
- Sample 100 random items that user hasn't interacted with for test set
- Set seed of 42

In [24]:
def sample_unrated_items_fast(user_item_sparse, user_ids, movie_ids, num_samples=100):
    rng = np.random.default_rng(42)
    num_users, num_items = user_item_sparse.shape
    rows, cols = [], []
    
    for u in range(num_users):
        start, end = user_item_sparse.indptr[u], user_item_sparse.indptr[u + 1]
        rated = user_item_sparse.indices[start:end]
        # Random sampling by rejection — fast when sparsity is high
        sampled = set()
        while len(sampled) < num_samples:
            candidates = rng.integers(0, num_items, size=num_samples * 2)
            for c in candidates:
                if c not in rated and c not in sampled:
                    sampled.add(c)
                    if len(sampled) == num_samples:
                        break
        rows.extend([user_ids[u]] * num_samples)
        cols.extend([movie_ids[s] for s in sampled])
    
    return pd.DataFrame({'userId': rows, 'movieId': cols})

Defined in cell 17

In [27]:
unrated_items = sample_unrated_items_fast(user_item_matrix, user_ids, movie_ids, num_samples=100)

In [28]:
unrated_items.head()

,userId,movieId
0,1,194212
1,1,100338
2,1,172433
3,1,181407
4,1,26197


### Concatenate with last interaction

In [31]:
last_reviews.head()

c:\Users\Atvar\miniconda3\envs\qc\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,userId,movieId,rating,timestamp
19851495,124331,109,1.0,825080160
6352965,39588,441,3.0,825638400
8114426,50823,446,5.0,825638400
5612144,35011,357,5.0,825638400
9415346,58832,509,5.0,825638400


In [32]:
unrated_items_with_labels = unrated_items.copy()
unrated_items_with_labels['label'] = 0  # unrated items get label 0

In [33]:
test_set = pd.concat([last_reviews[['userId', 'movieId', 'rating']].rename(columns={'rating': 'label'}), unrated_items_with_labels], ignore_index=True)

In [34]:
test_set.head()

,userId,movieId,label
0,124331,109,1.0
1,39588,441,3.0
2,50823,446,5.0
3,35011,357,5.0
4,58832,509,5.0


In [35]:
test_set.shape

(20295748, 3)

In [36]:
unrated_items.shape

(20094800, 2)

In [37]:
test_set.to_csv('../data/ml-32m/test_set.csv', index=False)